# Upload the thinned annotation pool to Roboflow

Uploads `MyDrive/Playbook/annotation_pool_25` to the **PLayers Detection** project (`players-detection-my09y`),
one batch per match, each image tagged with its match name, lighting and `pool25`. Images go in **unlabelled**
(no dataset split yet), ready for auto-labelling. `match_video_3` is skipped: its 75 frames are already in the
dataset.

Needs the Colab secret `ROBOFLOW_API_KEY` (key icon on the left, notebook access on). No GPU needed.

In [ ]:
# Upload the thinned pool to Roboflow (unlabelled, tagged per match; match_video_3 is skipped: it is already in)
!pip install -q roboflow
import os, time
from google.colab import drive, userdata
drive.mount('/content/drive')
from roboflow import Roboflow

POOL = '/content/drive/MyDrive/Playbook/annotation_pool_25'
WORKSPACE, PROJECT = 'muwafag-hussain-o1ghr', 'players-detection-my09y'
SKIP = {'match_video_3'}
LIGHTING = {'match_video_2': 'night', 'match_video_3': 'day', 'match_video_4': 'night', 'match_video_5': 'night',
            'match_video_6': 'night', 'match_video_7': 'night', 'match_video_8': 'night', 'match_video_9': 'dusk',
            'match_video_10': 'day', 'match_video_11': 'day', 'match_video_12': 'night'}

project = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(WORKSPACE).project(PROJECT)
done = failed = 0
for match in sorted(d for d in os.listdir(POOL) if os.path.isdir(f'{POOL}/{d}') and d not in SKIP):
    imgs = sorted(f for f in os.listdir(f'{POOL}/{match}') if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    tags = [match, LIGHTING.get(match, 'unknown'), 'pool25']
    for f in imgs:
        for attempt in range(3):
            try:
                project.upload(image_path=f'{POOL}/{match}/{f}', batch_name=f'pool25_{match}', tag_names=tags)
                done += 1
                break
            except Exception as e:
                if attempt == 2:
                    failed += 1; print('FAILED', f, e)
                time.sleep(2)
    print(f'{match}: {len(imgs)} images uploaded (tags {tags})', flush=True)
print(f'\nDone: {done} uploaded, {failed} failed. Tell Claude so it can check the project.')